In [22]:
CLASSIFIER_PATH = '../dataset/classifier/classifier.json'
MACHINES_CLASSIFIER_PATH = '../dataset/documents/Классификатор Версия №43.xlsx'

## Получение таблицы классфикатора машин и техники

In [23]:
from openpyxl import load_workbook
import polars as pl
import hashlib

workbook = load_workbook(MACHINES_CLASSIFIER_PATH)

def extract_name(s: str):
    try:
        return s.split(": ")[1]
    except Exception as e:
        print(s)
        raise e

SECTION_WORD = 'Раздел '
GROUP_WORD = 'Группа '

sheet = workbook['Машины и механизмы']

ACTIVE_SECTION = None
ACTIVE_GROUP = None

data = []
machines_set = set[str]()

for row in sheet.rows:
    row_content = " ".join([str(cell.value) for cell in row if cell.value is not None]).strip()

    if row_content == '':
        continue
    
    if SECTION_WORD in row_content:
        if ACTIVE_SECTION is not None:
            machines_set = set[str]()

        ACTIVE_SECTION = row_content
        ACTIVE_GROUP = None
        continue

    if GROUP_WORD in row_content:
        ACTIVE_GROUP = row_content
        continue

    if ACTIVE_SECTION is not None and ACTIVE_GROUP is not None:
        if row[1].value is None:
            continue

        if row[1].value in machines_set:
            continue
        
        
        machines_set.add(row[1].value)    

        data.append([
            hashlib.md5((ACTIVE_SECTION + ACTIVE_GROUP).encode()).hexdigest(),
            extract_name(ACTIVE_SECTION), 
            extract_name(ACTIVE_GROUP),
            row[0].value.strip(),
            row[1].value.strip()
        ])

machine_classifier_df = pl.DataFrame(data, schema=["ID_классификатора", "Раздел", "Группа", "Код", "Название"])

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/functools.py:889: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  return dispatch(args[0].__class__)(*args, **kw)


## Формирование таблицы работ и классификация машин

In [24]:
import ijson
import polars as pl

rows: list[dict] = []

def as_float(value):
    return None if value is None else float(value)

with open(CLASSIFIER_PATH, "rb") as f:
    for collection in ijson.items(f, "domains.item.collections.item", use_float=True):
        for table in collection.get("tables", []):
            for work in table.get("works", []):
                base = {
                    "domain_id": collection.get("domain_id"),
                    "domain_name": collection.get("domain_name"),
                    "kind": collection.get("kind") or table.get("kind"),
                    "document_code": collection.get("document_code"),
                    "collection_number": collection.get("number"),
                    "collection_name": collection.get("name"),
                    "department_number": table.get("department_number"),
                    "department": table.get("department"),
                    "section_number": table.get("section_number"),
                    "section": table.get("section"),
                    "subsection_number": table.get("subsection_number"),
                    "subsection": table.get("subsection"),
                    "table_class_id": table.get("class_id"),
                    "table_code": table.get("code"),
                    "table_name": table.get("name"),
                    "work_class_id": work.get("class_id"),
                    "work_code": work.get("code"),
                    "work_name": work.get("name"),
                    "work_group": work.get("group"),
                    "work_full_name": work.get("full_name"),
                    "unit": work.get("unit") or table.get("unit"),
                    "workers_hours": as_float(work.get("workers_hours")),
                    "workers_grade": as_float(work.get("workers_grade")),
                    "machinists_hours": as_float(work.get("machinists_hours")),
                    "commissioning_hours": as_float(work.get("commissioning_hours")),
                    "labor_hours": as_float(work.get("labor_hours")),
                    "machines_hours": as_float(work.get("machines_hours")),
                    "source_file": collection.get("source_file"),
                }
                machines = work.get("machines") or []
                if not machines:
                    rows.append(
                        {
                            **base,
                            "machine_code": None,
                            "machine_name": None,
                            "machine_hours": None,
                        }
                    )
                    continue
                for machine in machines:
                    rows.append(
                        {
                            **base,
                            "machine_code": machine.get("code"),
                            "machine_name": machine.get("name"),
                            "machine_hours": as_float(machine.get("hours")),
                        }
                    )

float_cols = [
    "workers_hours",
    "workers_grade",
    "machinists_hours",
    "commissioning_hours",
    "labor_hours",
    "machines_hours",
    "machine_hours",
]

works_machines_df = pl.DataFrame(
    rows,
    schema_overrides={col: pl.Float64 for col in float_cols},
)

In [25]:
from ast import alias

works_classifier_df = works_machines_df.select(
    pl.col("domain_name").alias("Сфера").replace("", None),
    pl.col("document_code").alias("Документ").replace("", None),
    pl.col("collection_name").alias("Сборник").replace("", None),
    pl.col("department").alias("Раздел").replace("", None),
    pl.col("section").alias("Секция").replace("", None),
    pl.col("subsection").alias("Подсекция").replace("", None),
    pl.col("table_class_id").alias("Код таблицы").replace("", None),
    pl.col("table_name").alias("Название таблицы").replace("", None),
    pl.col("work_class_id").alias("Код работы").replace("", None),
    pl.col("work_full_name").alias("Название работы").replace("", None),
    pl.col("unit").alias("Единица измерения").replace("", None),
    pl.col('workers_hours').alias("Трудозатраты рабочих (чел/ч)"),
    pl.col('machinists_hours').alias("Трудозатраты машинистов(чел/ч)"),
    pl.col('commissioning_hours').alias("Трудозатраты пусконаладки (чел/ч)"),
    pl.col('labor_hours').alias("Суммарные трудозатраты"),
    pl.col('machines_hours').alias("Машинное время (итог по норме)"),
    pl.col('machine_code').alias('Код машины').replace("", None),
    pl.col('machine_hours').alias('Машинное время (на норму)')
)

## Итоговые классификаторы

### Классификатор машин

In [ ]:
for col in machine_classifier_df.columns:
    print(f'{col}, тип: {machine_classifier_df[col].dtype}, нулевых: {machine_classifier_df[col].null_count()}')


ID_классификатора 0
Раздел 0
Группа 0
Код 0
Название 0


In [27]:
machine_classifier_df

ID_классификатора,Раздел,Группа,Код,Название
str,str,str,str,str
"""2449b59cdf42cfe81004951d4fd184…","""Машины для земляных работ""","""Бульдозеры""","""91.01.01-014""","""Бульдозеры-рыхлители на тракто…"
"""2449b59cdf42cfe81004951d4fd184…","""Машины для земляных работ""","""Бульдозеры""","""91.01.01-015""","""Бульдозеры-рыхлители на тракто…"
"""2449b59cdf42cfe81004951d4fd184…","""Машины для земляных работ""","""Бульдозеры""","""91.01.01-016""","""Бульдозеры-рыхлители на тракто…"
"""2449b59cdf42cfe81004951d4fd184…","""Машины для земляных работ""","""Бульдозеры""","""91.01.01-018""","""Бульдозеры-рыхлители на тракто…"
"""2449b59cdf42cfe81004951d4fd184…","""Машины для земляных работ""","""Бульдозеры""","""91.01.01-019""","""Бульдозеры-рыхлители на тракто…"
…,…,…,…,…
"""cd1cf7a1f868a6af74b11e1574ac17…","""Строительные машины и механизм…","""Машины для строительства магис…","""96.01.10-001""","""Трубоукладчики для труб диамет…"
"""6e078ffcf69f25ead3b5ac690094cd…","""Строительные машины и механизм…","""Тракторы, прицепы тракторные""","""96.01.15-001""","""Тракторы на гусеничном ходу, м…"
"""6e078ffcf69f25ead3b5ac690094cd…","""Строительные машины и механизм…","""Тракторы, прицепы тракторные""","""96.01.15-002""","""Тракторы на гусеничном ходу, м…"


### Классификатор работ

In [ ]:
for col in works_classifier_df.columns:
    print(f'{col}, тип: {works_classifier_df[col].dtype}, нулевых: {works_classifier_df[col].null_count()}')

Сфера, тип: String, нулевых: 0
Документ, тип: String, нулевых: 0
Сборник, тип: String, нулевых: 0
Раздел, тип: String, нулевых: 82786
Секция, тип: String, нулевых: 7364
Подсекция, тип: String, нулевых: 92884
Код таблицы, тип: String, нулевых: 0
Название таблицы, тип: String, нулевых: 0
Код работы, тип: String, нулевых: 0
Название работы, тип: String, нулевых: 0
Единица измерения, тип: String, нулевых: 0
Трудозатраты рабочих (чел/ч), тип: Float64, нулевых: 6771
Трудозатраты машинистов(чел/ч), тип: Float64, нулевых: 20819
Трудозатраты пусконаладки (чел/ч), тип: Float64, нулевых: 148124
Суммарные трудозатраты, тип: Float64, нулевых: 122
Машинное время (итог по норме), тип: Float64, нулевых: 11042
Код машины, тип: String, нулевых: 11042
Машинное время (на норму), тип: Float64, нулевых: 11042


In [29]:
works_classifier_df

Сфера,Документ,Сборник,Раздел,Секция,Подсекция,Код таблицы,Название таблицы,Код работы,Название работы,Единица измерения,Трудозатраты рабочих (чел/ч),Трудозатраты машинистов(чел/ч),Трудозатраты пусконаладки (чел/ч),Суммарные трудозатраты,Машинное время (итог по норме),Код машины,Машинное время (на норму)
str,str,str,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,str,f64
"""Капитальный ремонт оборудовани…","""ГЭСНмр 81-06-01-2022""","""Капитальный ремонт и модерниза…","""РАБОТЫ ПО ЗАМЕНЕ ОБОРУДОВАНИЯ,…",null,null,"""ГЭСНмр 01-01-001""","""Замена замка распашной двери ш…","""ГЭСНмр 01-01-001-01""","""Замена замка распашной двери ш…","""шт""",4.63,null,null,4.63,null,null,null
"""Капитальный ремонт оборудовани…","""ГЭСНмр 81-06-01-2022""","""Капитальный ремонт и модерниза…","""РАБОТЫ ПО ЗАМЕНЕ ОБОРУДОВАНИЯ,…",null,null,"""ГЭСНмр 01-01-001""","""Замена замка распашной двери ш…","""ГЭСНмр 01-01-001-02""","""Замена замка распашной двери ш…","""шт""",6.19,null,null,6.19,null,null,null
"""Капитальный ремонт оборудовани…","""ГЭСНмр 81-06-01-2022""","""Капитальный ремонт и модерниза…","""РАБОТЫ ПО ЗАМЕНЕ ОБОРУДОВАНИЯ,…",null,null,"""ГЭСНмр 01-01-002""","""Замена оборудования главного п…","""ГЭСНмр 01-01-002-01""","""Замена электродвигателя, колич…","""шт""",26.2,0.7,null,26.9,1.2,"""91.17.04-233""",1.2
"""Капитальный ремонт оборудовани…","""ГЭСНмр 81-06-01-2022""","""Капитальный ремонт и модерниза…","""РАБОТЫ ПО ЗАМЕНЕ ОБОРУДОВАНИЯ,…",null,null,"""ГЭСНмр 01-01-002""","""Замена оборудования главного п…","""ГЭСНмр 01-01-002-02""","""Замена электродвигателя, колич…","""шт""",20.6,0.7,null,21.3,null,null,null
"""Капитальный ремонт оборудовани…","""ГЭСНмр 81-06-01-2022""","""Капитальный ремонт и модерниза…","""РАБОТЫ ПО ЗАМЕНЕ ОБОРУДОВАНИЯ,…",null,null,"""ГЭСНмр 01-01-002""","""Замена оборудования главного п…","""ГЭСНмр 01-01-002-03""","""Замена полумуфты электродвигат…","""шт""",17.7,3.6,null,21.3,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Строительные и специальные стр…","""ГЭСН 81-02-47-2022""","""Озеленение, защитные лесонасаж…",null,"""ЗАЩИТНЫЕ ЛЕСОНАСАЖДЕНИЯ""","""ПРОЧИЕ РАБОТЫ""","""ГЭСН 47-02-094""","""Выкашивание травы и срезка пор…","""ГЭСН 47-02-094-02""","""Выкашивание травы вручную при …","""га""",11.7,null,null,11.7,null,null,null
"""Строительные и специальные стр…","""ГЭСН 81-02-47-2022""","""Озеленение, защитные лесонасаж…",null,"""ЗАЩИТНЫЕ ЛЕСОНАСАЖДЕНИЯ""","""ПРОЧИЕ РАБОТЫ""","""ГЭСН 47-02-094""","""Выкашивание травы и срезка пор…","""ГЭСН 47-02-094-03""","""Выкашивание травы вручную при …","""га""",4.34,null,null,4.34,null,null,null
"""Строительные и специальные стр…","""ГЭСН 81-02-47-2022""","""Озеленение, защитные лесонасаж…",null,"""ЗАЩИТНЫЕ ЛЕСОНАСАЖДЕНИЯ""","""ПРОЧИЕ РАБОТЫ""","""ГЭСН 47-02-094""","""Выкашивание травы и срезка пор…","""ГЭСН 47-02-094-04""","""Выкашивание травы вручную при …","""га""",null,null,null,null,null,null,null
